#Install necessary libraries




In [7]:
! pip install -q transformers
print("steup complete.")

steup complete.


#step -01 load small language model


In [9]:
!pip install -q --force-reinstall pillow

from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
print("GPT-2 loaded. Vocabulary size:", tokenizer.vocab_size)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded. Vocabulary size: 50257


#step -02 See the probability for just the next word

In [18]:
prompt = "The best way to build a strong project"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(input_ids)
    next_token_logits = outputs.logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim=-1)

top5 = torch.topk(next_token_probs, 5)
print(f'Prompt: "{prompt}"')
print("Top 5 candidates for the NEXT single word:\n")
for prob, idx in zip(top5.values, top5.indices):
    word = tokenizer.decode([idx])
    print(f'   "{word}"   probability = {prob.item():.4f}')

Prompt: "The best way to build a strong project"
Top 5 candidates for the NEXT single word:

   " is"   probability = 0.3726
   ","   probability = 0.0445
   " team"   probability = 0.0359
   " with"   probability = 0.0253
   " in"   probability = 0.0250


step - 03 Build autoreggresive loop by hand


In [19]:
def generate_step_by_step(prompt, num_steps=5):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    generated = input_ids

    for step in range(num_steps):
        with torch.no_grad():
            outputs = model(generated)
            next_token_logits = outputs.logits[0, -1, :]

        # Pick the single highest-probability next token (this is GREEDY decoding)
        next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)

        # THE CORE LOOP: append the new token, feed the WHOLE thing back in
        generated = torch.cat((generated, next_token_id), dim=-1)

        chosen_word = tokenizer.decode(next_token_id[0])
        so_far = tokenizer.decode(generated[0])
        print(f"Step {step+1}: picked \"{chosen_word}\" -> so far: \"{so_far}\"")

    return tokenizer.decode(generated[0])

final_text = generate_step_by_step("The best way to build a strong project", num_steps=5)
print("\nFINAL:", final_text)

Step 1: picked " is" -> so far: "The best way to build a strong project is"
Step 2: picked " to" -> so far: "The best way to build a strong project is to"
Step 3: picked " build" -> so far: "The best way to build a strong project is to build"
Step 4: picked " it" -> so far: "The best way to build a strong project is to build it"
Step 5: picked " from" -> so far: "The best way to build a strong project is to build it from"

FINAL: The best way to build a strong project is to build it from


#Compare to this one line shortcut

In [21]:
input_ids = tokenizer.encode("The best way to build a strong project", return_tensors="pt")
outputs = model.generate(input_ids, max_new_tokens=7, do_sample=False)
print(tokenizer.decode(outputs[0]))


The best way to build a strong project is to build it from the ground
